# 🧠 Nephro-AI — PDF Extractor (Google Colab)

This notebook processes large medical PDFs using **Docling** (GPU-accelerated OCR + layout parsing)  
and outputs `*_chunks.json` + `*_metadata.json` files into Google Drive.

After it finishes, copy the output folder → `ai-engine/data/processed/` on your local machine  
then run `prepare_vectordb.py` and `build_vectordb.py` locally.

---

## ✅ Before Running

1. **Change runtime to GPU**: `Runtime → Change runtime type → T4 GPU`
2. **Upload source files & PDFs to Google Drive** (see Cell 2 for folder structure)
3. **Fill in your API keys** in Cell 4

## 📁 Required Google Drive folder structure

```
MyDrive/
└── NephroAI/
    ├── src/
    │   ├── pdf_extractor.py
    │   ├── config.py
    │   ├── document_tracker.py
    │   └── openai_embeddings.py
    └── raw_pdfs/
        ├── your_large_book.pdf
        └── another_paper.pdf
```

---
## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

---
## Step 2 — Verify Drive folder structure

⚠️ Edit `DRIVE_ROOT` below if your folder is named differently.

In [ ]:
import os

# ─── CONFIGURE THESE PATHS ────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/NephroAI'   # Root folder in your Drive
DRIVE_SRC    = f'{DRIVE_ROOT}/src'                  # Folder containing .py source files
DRIVE_PDFS   = f'{DRIVE_ROOT}/raw_pdfs'             # Folder containing new large PDFs
# ──────────────────────────────────────────────────────────────────────────

print('Checking Drive structure...')
for path in [DRIVE_ROOT, DRIVE_SRC, DRIVE_PDFS]:
    exists = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {exists}  {path}')

print()
print('Source files in Drive/src:')
if os.path.exists(DRIVE_SRC):
    for f in sorted(os.listdir(DRIVE_SRC)):
        print(f'  📄 {f}')
else:
    print('  ❌ src folder not found — upload your .py files first!')

print()
print('PDFs in Drive/raw_pdfs:')
if os.path.exists(DRIVE_PDFS):
    pdfs = [f for f in os.listdir(DRIVE_PDFS) if f.lower().endswith('.pdf')]
    for f in sorted(pdfs):
        size_mb = os.path.getsize(f'{DRIVE_PDFS}/{f}') / (1024*1024)
        print(f'  📕 {f}  ({size_mb:.1f} MB)')
    print(f'\n  Total: {len(pdfs)} PDF(s)')
else:
    print('  ❌ raw_pdfs folder not found — upload your PDFs first!')

---
## Step 3 — Install all dependencies

⏳ This takes **3–5 minutes** (Docling with OCR is large). Run once per session.

In [ ]:
print('Installing base packages...')
!pip install -q PyPDF2 pdfplumber langdetect nltk tqdm python-dotenv numpy
print('✅ Base packages installed')

# Fix: upgrade typer BEFORE installing docling to prevent
# the "typer-slim requires typer>=0.24.0" dependency conflict
print('Fixing typer version conflict...')
!pip install -q --upgrade typer
print('✅ typer upgraded')

# Fix: docling v2.x no longer uses the [ocr] extra — OCR is built in.
# Installing 'docling[ocr]' on v2.76.0+ raises:
#   WARNING: docling 2.76.x does not provide the extra 'ocr'
# Just install 'docling' directly.
print('Installing Docling (OCR is built-in since v2.x)...')
!pip install -q docling
print('✅ Docling installed')

print('Installing Google GenAI SDK...')
!pip install -q google-generativeai
print('✅ Google GenAI installed')

# Download NLTK data
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✅ NLTK data downloaded')

# Verify docling + OCR pipeline imports correctly
print()
print('Verifying Docling installation...')
try:
    from docling.document_converter import DocumentConverter
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    print('✅ Docling import OK — ready for GPU-accelerated extraction')
except ImportError as e:
    print(f'❌ Docling import failed: {e}')
    print('   Try: !pip install --upgrade docling')

print()
print('All dependencies ready!')

---
## Step 4 — Set API Keys

⚠️ Fill in your actual API keys below.

In [ ]:
import os

# ─── FILL IN YOUR API KEYS ────────────────────────────────────────────────
GOOGLE_API_KEY     = 'YOUR_GOOGLE_GEMINI_API_KEY'    # From https://aistudio.google.com
OPENROUTER_API_KEY = 'YOUR_OPENROUTER_API_KEY'       # From https://openrouter.ai
# ──────────────────────────────────────────────────────────────────────────

os.environ['GOOGLE_API_KEY']     = GOOGLE_API_KEY
os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY

if GOOGLE_API_KEY == 'YOUR_GOOGLE_GEMINI_API_KEY':
    print('⚠️  WARNING: GOOGLE_API_KEY is not set. VLM image captioning will fail.')
else:
    print('✅ GOOGLE_API_KEY set')

if OPENROUTER_API_KEY == 'YOUR_OPENROUTER_API_KEY':
    print('⚠️  WARNING: OPENROUTER_API_KEY is not set. Semantic sub-chunking may fail.')
else:
    print('✅ OPENROUTER_API_KEY set')

---
## Step 5 — Set up Colab folder structure & copy source files

In [ ]:
import os
import shutil

# ── Colab working directories ────────────────────────────────────────────
COLAB_ROOT       = '/content/nephro_ai'
COLAB_SRC        = f'{COLAB_ROOT}/src/chatbot'
COLAB_RAW        = f'{COLAB_ROOT}/data/raw'
COLAB_PROCESSED  = f'{COLAB_ROOT}/data/processed'
COLAB_MANIFEST   = f'{COLAB_PROCESSED}/processed_manifest.json'

for path in [COLAB_SRC, COLAB_RAW, COLAB_PROCESSED]:
    os.makedirs(path, exist_ok=True)

# ── Create empty __init__.py files so Python treats dirs as packages ─────
for pkg in [f'{COLAB_ROOT}/src', COLAB_SRC]:
    init = f'{pkg}/__init__.py'
    if not os.path.exists(init):
        open(init, 'w').close()

# ── Copy source files from Drive → Colab ────────────────────────────────
required_files = ['pdf_extractor.py', 'config.py', 'document_tracker.py', 'openai_embeddings.py']
print('Copying source files from Drive...')
for fname in required_files:
    src  = f'{DRIVE_SRC}/{fname}'
    dest = f'{COLAB_SRC}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, dest)
        print(f'  ✅ {fname}')
    else:
        print(f'  ❌ {fname} — NOT FOUND in {DRIVE_SRC}')

# ── Copy PDFs from Drive → Colab ────────────────────────────────────────
print()
print('Copying PDFs from Drive...')
pdfs = [f for f in os.listdir(DRIVE_PDFS) if f.lower().endswith('.pdf')]
for pdf in pdfs:
    src  = f'{DRIVE_PDFS}/{pdf}'
    dest = f'{COLAB_RAW}/{pdf}'
    if not os.path.exists(dest):  # Skip if already copied
        shutil.copy2(src, dest)
        size_mb = os.path.getsize(dest) / (1024*1024)
        print(f'  📕 {pdf}  ({size_mb:.1f} MB)')
    else:
        print(f'  ⏭️  {pdf} already copied, skipping')

print(f'\n✅ Ready: {len(pdfs)} PDF(s) in {COLAB_RAW}')

---
## Step 6 — Patch config paths to use Colab directories

In [ ]:
import sys
from pathlib import Path

# Add Colab source directories to Python path
for path in [f'{COLAB_ROOT}/src', f'{COLAB_ROOT}/src/chatbot']:
    if path not in sys.path:
        sys.path.insert(0, path)

# Import config and patch all paths to Colab directories
import chatbot.config as cfg

cfg.PROJECT_ROOT       = Path(COLAB_ROOT)
cfg.DATA_DIR           = Path(f'{COLAB_ROOT}/data')
cfg.RAW_DATA_DIR       = Path(COLAB_RAW)
cfg.PROCESSED_DATA_DIR = Path(COLAB_PROCESSED)
cfg.VECTORDB_READY_DIR = Path(f'{COLAB_ROOT}/data/vectordb_ready/documents')
cfg.VECTORDB_DIR       = Path(f'{COLAB_ROOT}/vectordb')
cfg.CHROMA_DB_PATH     = Path(f'{COLAB_ROOT}/vectordb/chroma_db')

# Also patch SINHALA_MED_DICT_PATH to avoid FileNotFoundError
cfg_module_path = Path(f'{COLAB_SRC}/config.py')
cfg.SINHALA_MED_DICT_PATH = Path(f'{COLAB_ROOT}/data/sinhala_med_dict.json')

# Ensure directories exist
for d in [cfg.PROCESSED_DATA_DIR, cfg.VECTORDB_READY_DIR, cfg.VECTORDB_DIR, cfg.CHROMA_DB_PATH]:
    d.mkdir(parents=True, exist_ok=True)

print('Config paths patched:')
print(f'  RAW_DATA_DIR       = {cfg.RAW_DATA_DIR}')
print(f'  PROCESSED_DATA_DIR = {cfg.PROCESSED_DATA_DIR}')
print(f'  VECTORDB_READY_DIR = {cfg.VECTORDB_READY_DIR}')
print(f'  CHROMA_DB_PATH     = {cfg.CHROMA_DB_PATH}')
print()

# Verify Docling is available
try:
    from docling.document_converter import DocumentConverter
    print('✅ Docling is available and ready (GPU-accelerated)')
except ImportError:
    print('❌ Docling not found — re-run Step 3')

---
## Step 7 — Check GPU availability

In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:')
    # Print just the GPU name line
    for line in result.stdout.split('\n'):
        if 'Tesla' in line or 'T4' in line or 'A100' in line or 'V100' in line or 'L4' in line:
            print(f'   {line.strip()}')
    print()
    print('Docling OCR will use the GPU — significantly faster than CPU!')
else:
    print('⚠️  No GPU detected!')
    print('   ➜ Go to Runtime → Change runtime type → Hardware accelerator → T4 GPU')
    print('   ➜ Then restart and re-run all cells from the beginning.')

---
## Step 8 — Run PDF Extraction

This is the main processing step. It will:
- Use **Docling** with dynamic OCR routing (GPU-accelerated)
- Auto-detect native vs scanned pages and only OCR scanned ones
- Skip already-processed files (SHA-256 deduplication)
- Save `*_chunks.json` + `*_metadata.json` files to the processed folder

⏳ Processing time depends on PDF size. ~1–3 min per 100 pages with GPU.

In [ ]:
import os
import sys

# Re-ensure paths are in sys.path
for path in [f'{COLAB_ROOT}/src', f'{COLAB_ROOT}/src/chatbot']:
    if path not in sys.path:
        sys.path.insert(0, path)

from chatbot.pdf_extractor import PDFKnowledgeExtractor
from chatbot.document_tracker import DocumentTracker
import chatbot.config as cfg

# Configuration — matches your local config.py settings
OUTPUT_DIR  = str(cfg.PROCESSED_DATA_DIR)
CHUNK_SIZE  = cfg.get_chunk_config().get('max_words', 600)
OVERLAP     = cfg.get_chunk_config().get('overlap_sentences', 2) * 10

# Scan all PDFs in the raw directory
file_paths = sorted([
    os.path.join(COLAB_RAW, f)
    for f in os.listdir(COLAB_RAW)
    if f.lower().endswith('.pdf') or f.lower().endswith('.txt')
])

if not file_paths:
    print('❌ No PDF files found in', COLAB_RAW)
    print('   Re-run Step 5 to copy PDFs from Drive.')
else:
    print(f'Found {len(file_paths)} file(s) to process:')
    for f in file_paths:
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f'  📕 {os.path.basename(f)}  ({size_mb:.1f} MB)')

    print()
    print(f'Output directory : {OUTPUT_DIR}')
    print(f'Chunk size       : {CHUNK_SIZE} words')
    print(f'Overlap          : {OVERLAP} words')
    print()

In [ ]:
# ── MAIN EXTRACTION LOOP ─────────────────────────────────────────────────
# Uses SHA-256 dedup tracker — safe to re-run if session crashes

tracker    = DocumentTracker(manifest_path=COLAB_MANIFEST)
results    = []
successful = 0
failed     = 0
skipped    = 0

for idx, file_path in enumerate(file_paths, 1):
    print()
    print('=' * 70)
    print(f' PROCESSING FILE {idx}/{len(file_paths)}: {os.path.basename(file_path)}')
    print('=' * 70)

    # Skip already-processed files (crash-safe resume)
    if tracker.is_already_processed(file_path):
        print(f' ⏭️  SKIP: Already processed (SHA-256 match in manifest)')
        skipped += 1
        results.append({'input': file_path, 'status': 'skipped'})
        continue

    try:
        extractor = PDFKnowledgeExtractor(file_path, OUTPUT_DIR)
        output_file = extractor.process(
            chunk_size=CHUNK_SIZE,
            overlap=OVERLAP,
            save_format='json'
        )

        if output_file:
            successful += 1
            tracker.mark_as_processed(file_path, chunk_count=len(extractor.chunks))
            results.append({
                'input':         file_path,
                'output':        output_file,
                'status':        'success',
                'method':        extractor.metadata.get('extraction_method', 'unknown'),
                'chunks':        len(extractor.chunks),
                'pages':         extractor.metadata.get('total_pages', '?'),
                'native_pages':  extractor.metadata.get('native_pages', '?'),
                'scanned_pages': extractor.metadata.get('scanned_pages', '?'),
                'chars':         extractor.metadata.get('raw_text_length', 0),
                'format':        extractor.extraction_format,
            })
            print(f'\n ✅ Done: {len(extractor.chunks)} chunks → {os.path.basename(output_file)}')
        else:
            failed += 1
            results.append({'input': file_path, 'status': 'failed', 'method': extractor.metadata.get('extraction_method', '?')})
            print(' ❌ Processing returned None — file may be corrupt or empty')

    except Exception as e:
        failed += 1
        results.append({'input': file_path, 'status': 'error', 'error': str(e)})
        print(f' ❌ ERROR: {e}')
        import traceback
        traceback.print_exc()

# ── SUMMARY ──────────────────────────────────────────────────────────────
print()
print('=' * 70)
print(' BATCH COMPLETE')
print('=' * 70)
print(f'  Total    : {len(file_paths)}')
print(f'  ✅ Done   : {successful}')
print(f'  ⏭️  Skipped: {skipped}')
print(f'  ❌ Failed : {failed}')
print()
print('Per-file results:')
for r in results:
    name   = os.path.basename(r['input'])
    status = r['status']
    if status == 'success':
        print(f'  ✅ {name}  →  {r["chunks"]} chunks  [{r["method"]}]')
    elif status == 'skipped':
        print(f'  ⏭️  {name}  (already processed)')
    else:
        print(f'  ❌ {name}  →  {r.get("error", "failed")}')

---
## Step 9 — Verify output files

In [ ]:
import os

output_files = sorted(os.listdir(COLAB_PROCESSED))
chunks_files   = [f for f in output_files if f.endswith('_chunks.json')]
metadata_files = [f for f in output_files if f.endswith('_metadata.json')]
manifest_files = [f for f in output_files if f.endswith('.json') and 'manifest' in f]

print(f'Output folder: {COLAB_PROCESSED}')
print(f'  _chunks.json files   : {len(chunks_files)}')
print(f'  _metadata.json files : {len(metadata_files)}')
print(f'  manifest file        : {len(manifest_files)}')
print()

total_size_mb = sum(
    os.path.getsize(os.path.join(COLAB_PROCESSED, f)) for f in output_files
) / (1024 * 1024)
print(f'Total output size: {total_size_mb:.1f} MB')
print()

print('Files generated:')
for f in output_files:
    size_kb = os.path.getsize(os.path.join(COLAB_PROCESSED, f)) / 1024
    print(f'  📄 {f}  ({size_kb:.0f} KB)')

---
## Step 10 — Zip output and save to Google Drive

In [ ]:
import shutil
import os
from datetime import datetime

# Create timestamped zip filename
timestamp  = datetime.now().strftime('%Y%m%d_%H%M')
zip_name   = f'processed_chunks_{timestamp}'
zip_local  = f'/content/{zip_name}'
zip_drive  = f'{DRIVE_ROOT}/{zip_name}.zip'

print(f'Zipping {COLAB_PROCESSED} ...')
shutil.make_archive(zip_local, 'zip', COLAB_PROCESSED)
zip_size_mb = os.path.getsize(f'{zip_local}.zip') / (1024 * 1024)
print(f'  ✅ Zip created: {zip_local}.zip  ({zip_size_mb:.1f} MB)')

print(f'Copying to Google Drive: {zip_drive} ...')
shutil.copy2(f'{zip_local}.zip', zip_drive)
print(f'  ✅ Saved to Drive!')

print()
print('=' * 60)
print(' DONE!')
print('=' * 60)
print(f'  Download from Drive: NephroAI/{zip_name}.zip')
print()
print('  Next steps on your LOCAL machine:')
print('  1. Download and extract the zip')
print('  2. Copy all *_chunks.json and *_metadata.json files into:')
print('     ai-engine/data/processed/')
print('  3. Run: python src/chatbot/prepare_vectordb.py')
print('  4. Run: python src/chatbot/build_vectordb.py')
print('=' * 60)

---
## (Optional) Step 11 — Download zip directly to your browser

If you prefer downloading directly instead of going through Drive:

In [ ]:
from google.colab import files
import glob

# Find the most recent zip
zips = sorted(glob.glob('/content/processed_chunks_*.zip'), reverse=True)
if zips:
    latest = zips[0]
    size_mb = os.path.getsize(latest) / (1024*1024)
    print(f'Downloading {os.path.basename(latest)} ({size_mb:.1f} MB)...')
    files.download(latest)
else:
    print('No zip found — run Step 10 first')